#### Set environment variables in [.env](.env) for LLM API calling

### Import Dependencies

In [ ]:
import sys
import os
sys.path.insert(0, "../")
import promptwizard
import random
from promptwizard.glue.promptopt.instantiate import GluePromptOpt
from promptwizard.glue.promptopt.techniques.common_logic import DatasetSpecificProcessing
from promptwizard.glue.common.utils.file import save_jsonlist
from typing import Any
from tqdm import tqdm
import json
import os
from azure.identity import get_bearer_token_provider, AzureCliCredential
from openai import AzureOpenAI
from openai import OpenAI

from dotenv import load_dotenv
load_dotenv(override = True)


### Below code can be used for LLM-as-a-judge eval

In [ ]:
# extract_between / llm_eval / BBH now live in bbh_processor.py, shared across
# every optimizer notebook (PromptWizard, APE, ProTeGi, MPIR) to keep the
# exact-match default and LLM-judge retarget in one place.
from bbh_processor import BBH, LLM_AS_JUDGE_EVAL

### Create a dataset specific class and define the required functions 

In [ ]:
bbh_processor = BBH()

### Load and save the dataset . 
Set the ```dataset_to_run``` variable to choose 1 among the 19 datasets of BBII to run the optimization on

In [ ]:
# Step 1: Prepare directories
if not os.path.exists("data"):
    os.mkdir("data")

dataset_to_run = 'reasoning_about_colored_objects'
seed = 42

# Shared, idempotent three-way split (data_prep.py) -- every optimizer
# notebook for this (task, seed) gets the identical train/mpir-validation/test
# partition, which is required for the example-level paired analysis in
# REBUILD.md §3.2 to compare like with like. PromptWizard's own optimization
# only consumes the optimizer-train file; the mpir-validation file is written
# but unused here, reserved for a later MPIR pass over this run's output.
from data_prep import prepare_bbh_task_split

split_paths = prepare_bbh_task_split(dataset_to_run, bbh_processor, seed=seed)


### Set paths

In [ ]:
train_file_name = split_paths.train_file_name
test_file_name = split_paths.test_file_name
path_to_config = "configs"
llm_config_path = os.path.join(path_to_config, "promptwizard/llm_config.yaml")
promptopt_config_path = os.path.join(path_to_config, "promptwizard/promptopt_config_reasoning_about_colored_objects.yaml")
setup_config_path = os.path.join(path_to_config, "promptwizard/setup_config.yaml")

### Create an object for calling prompt optimization and inference functionalities

In [ ]:
gp = GluePromptOpt(promptopt_config_path,
                   setup_config_path,
                   train_file_name,
                   bbh_processor,
                   seed=seed)

### Call prompt optmization function
1. ```use_examples``` can be used when there are training samples and a mixture of real and synthetic in-context examples are required in the final prompt. When set to ```False``` all the in-context examples will be real
2. ```generate_synthetic_examples``` can be used when there are no training samples and we want to generate synthetic examples 
3. ```run_without_train_examples``` can be used when there are no training samples and in-context examples are not required in the final prompt 

In [ ]:
# Function call to generate optimal prompt and expert profile 
best_prompt = gp.get_best_prompt(use_examples=False,run_without_train_examples=False,generate_synthetic_examples=False)

### Save the optimized prompt and expert profile

In [ ]:
print(best_prompt[0])

In [ ]:
import pickle
if not os.path.exists("results"):
    os.system("mkdir results")

# Task+seed-scoped filename: MPIR.ipynb constructs this same path from its own
# dataset_to_run/seed variables when loading, so a mismatched task/seed
# between the two notebooks raises FileNotFoundError instead of silently
# refining and evaluating the wrong task's prompt.
pkl_path = f"results/promptwizard_{dataset_to_run}_seed{seed}.pkl"
with open(pkl_path, 'wb') as f:
    pickle.dump(best_prompt[0], f)

print(f"Best prompt (saved to {pkl_path}): {best_prompt[0]}")

### Evaluate the optimized prompt

In [ ]:
gp.BEST_PROMPT = best_prompt[0]

# Function call to evaluate the prompt. task_name/condition_name/seed feed the
# tracked results/predictions/<task>_<condition>_<seed>.jsonl file (REBUILD.md §5).
accuracy = gp.evaluate(test_file_name, task_name=dataset_to_run,
                       condition_name="promptwizard", seed=seed)

print(f"Final Accuracy: {accuracy}")